# Deep Structural Decomposition of LSC Circuits

## Overview
Phase 2 found that **aggregate structural metrics** (size, component ratios, skip fraction) **do not differ by frequency band**, but **Jaccard within-band > between-band** for all 4 models. This means circuits differ in *which specific edges* they include, not in summary statistics.

This notebook decomposes the structure at finer granularities to locate **WHERE** the differences live.

## Hypotheses Addressed
- **H2**: Constant-size core-periphery architecture (shared universal core + band-specific periphery)
- **H5**: Subset mechanism: low-freq circuits contain high-freq circuits
- **H6**: Band-specific edges concentrate in MLPs and late layers
- **H7**: Structural similarity follows frequency-distance gradient
- **H9**: Frequency-sensitive layers form a contiguous zone

## Analysis Sections
1. Component-Level Decomposition (1a-1c)
2. Universal vs. Band-Specific Edge Characterization (2a-2d)
3. Layer-Level Decomposition (3a-3c)
4. Head-Level Decomposition (4a-4c)
5. Graph-Theoretic Complexity (5a-5d)
6. Input/Output Edge Analysis (6a-6b)
7. Draw Stability & Cross-Analyses (7a-7e)

## Data
- 60 circuits: 4 models x 5 bands x 3 draws
- Source: `all_circuits_structure.json`

## Note on Pythia's Parallel Architecture
Pythia uses parallel transformer blocks where attention and MLP run simultaneously (same layer index). Same-layer attn->mlp edges represent parallel->merge, not sequential processing.

---
## 0. Setup & Data Loading

In [1]:
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from collections import defaultdict
from pathlib import Path

# Add utils to path
sys.path.insert(0, str(Path.cwd()))

from utils import (
    # Constants
    MODELS,
    BANDS,
    FREQUENCY_BANDS,
    DRAWS,
    MODEL_INFO,
    BAND_NAMES,
    BAND_COLORS,
    MODEL_COLORS,
    COMPONENT_COLORS,
    FREQUENCY_RANK,
    ANALYSIS_DIR,
    VIZ_DIR,
    get_output_dirs,
    # Data loading
    load_extracted_data,
    # Edge analysis (original)
    get_edge_set,
    compute_jaccard,
    compute_universal_edges_per_draw,
    compute_band_specific_edges,
    compute_edge_sharing_spectrum,
    # Edge analysis (new deep decomposition)
    build_edge_index,
    get_filtered_edge_set,
    compute_component_jaccard,
    compute_edge_sharing_with_properties,
    compute_band_affinity_matrix,
    compute_band_affinity_summary,
    compute_layer_band_sensitivity,
    compute_per_layer_universal_fraction,
    compute_head_band_presence,
    compute_head_band_entropy,
    compute_directed_containment,
    # Draw stability (S-G3)
    compute_edge_draw_stability,
    compute_draw_stability_vs_sharing,
    # Graph analysis
    build_circuit_graph,
    compute_graph_metrics,
    compute_degree_stats,
    compute_hub_nodes,
    compute_all_graph_metrics,
    classify_hub_universality,
    # Universal core connectivity (S-G4)
    compute_universal_core_connectivity,
    # Plotting
    setup_plotting,
    save_figure,
    plot_component_jaccard_comparison,
    plot_layer_sensitivity_profile,
    plot_edge_sharing_profile,
    plot_band_affinity_heatmap,
    plot_head_universality_map,
    plot_graph_metrics_comparison,
    plot_layer_flow_heatmap,
)

# Setup
setup_plotting()
ANALYSIS_DIR, VIZ_DIR = get_output_dirs()
print(f"Analysis output: {ANALYSIS_DIR}")
print(f"Visualization output: {VIZ_DIR}")

Analysis output: LSC_circuit_analysis/02_Phase_Structural/outputs/analysis
Visualization output: LSC_circuit_analysis/02_Phase_Structural/outputs/viz


In [2]:
# Load all circuit data
circuits, df = load_extracted_data()
print(f"Loaded {len(circuits)} circuits")
print(f"Models: {df['model'].unique().tolist()}")
print(f"Bands: {df['band'].unique().tolist()}")
print(f"Draws per modelxband: {df.groupby(['model', 'band']).size().unique().tolist()}")

# Pre-build edge indices for all circuits
edge_indices = {}
for cid, c in circuits.items():
    edge_indices[cid] = build_edge_index(c)
print(f"Built edge indices for {len(edge_indices)} circuits")

Loaded 75 circuits (0 failed)
Loaded 75 circuits
Models: ['pythia-70m', 'pythia-160m', 'pythia-410m', 'pythia-1b', 'pythia-1.4b']
Bands: ['low', 'medium', 'high', 'very_high', 'control']
Draws per modelxband: [3]
Built edge indices for 75 circuits


---
## 1. Component-Level Decomposition

The existing analysis shows aggregate component ratios (attn%, mlp%, resid%) don't differ by band. But **which component type carries the band-specific edge variation** detected by Jaccard?

### 1a. Per-Component Jaccard

Compute within-band vs between-band Jaccard separately for attention, MLP, and residual edges. If variation concentrates in one component, its within-between gap will be largest.

In [3]:
# Compute per-component Jaccard for all models
component_types = ["attn", "mlp", "resid"]
comp_jaccard_rows = []

for model in MODELS:
    print(f"\n{model}:")
    for comp in component_types:
        result = compute_component_jaccard(circuits, model, comp)
        w_mean = np.mean(result["within"]) if result["within"] else 0
        b_mean = np.mean(result["between"]) if result["between"] else 0
        gap = w_mean - b_mean
        print(
            f"  {comp:6s}: within={w_mean:.4f}, between={b_mean:.4f}, gap={gap:+.4f} (n_w={len(result['within'])}, n_b={len(result['between'])})"
        )

        comp_jaccard_rows.append(
            {
                "model": model,
                "component": comp,
                "within_mean": w_mean,
                "within_std": np.std(result["within"]) if result["within"] else 0,
                "between_mean": b_mean,
                "between_std": np.std(result["between"]) if result["between"] else 0,
                "gap": gap,
                "n_within": len(result["within"]),
                "n_between": len(result["between"]),
                "within_values": result["within"],
                "between_values": result["between"],
            }
        )

df_comp_jaccard = pd.DataFrame(comp_jaccard_rows)
df_comp_jaccard.drop(columns=["within_values", "between_values"]).to_csv(
    ANALYSIS_DIR / "deep_component_jaccard.csv", index=False
)
print(f"\nSaved: {ANALYSIS_DIR / 'deep_component_jaccard.csv'}")


pythia-70m:
  attn  : within=0.6992, between=0.6569, gap=+0.0423 (n_w=30, n_b=90)
  mlp   : within=0.9245, between=0.9102, gap=+0.0144 (n_w=30, n_b=90)
  resid : within=0.9846, between=0.9712, gap=+0.0135 (n_w=30, n_b=90)

pythia-160m:
  attn  : within=0.4778, between=0.4400, gap=+0.0378 (n_w=30, n_b=90)
  mlp   : within=0.7147, between=0.6955, gap=+0.0192 (n_w=30, n_b=90)
  resid : within=0.9575, between=0.9284, gap=+0.0291 (n_w=30, n_b=90)

pythia-410m:


  attn  : within=0.3444, between=0.3278, gap=+0.0167 (n_w=30, n_b=90)


  mlp   : within=0.5520, between=0.5402, gap=+0.0117 (n_w=30, n_b=90)
  resid : within=0.8693, between=0.8444, gap=+0.0249 (n_w=30, n_b=90)

pythia-1b:
  attn  : within=0.4206, between=0.4066, gap=+0.0140 (n_w=30, n_b=90)
  mlp   : within=0.5361, between=0.5259, gap=+0.0102 (n_w=30, n_b=90)
  resid : within=0.7858, between=0.7772, gap=+0.0085 (n_w=30, n_b=90)

pythia-1.4b:


  attn  : within=0.3206, between=0.3030, gap=+0.0176 (n_w=30, n_b=90)
  mlp   : within=0.4565, between=0.4373, gap=+0.0191 (n_w=30, n_b=90)
  resid : within=0.7436, between=0.7143, gap=+0.0293 (n_w=30, n_b=90)



Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/analysis/deep_component_jaccard.csv


In [4]:
# Visualization: Per-component Jaccard comparison
fig, axes = plt.subplots(1, len(MODELS), figsize=(5 * len(MODELS), 5), sharey=True)

for idx, model in enumerate(MODELS):
    model_data = {
        row["component"]: {
            "within": row["within_values"],
            "between": row["between_values"],
        }
        for _, row in df_comp_jaccard[df_comp_jaccard["model"] == model].iterrows()
    }

    plot_component_jaccard_comparison(model_data, model, ax=axes[idx])
    if idx > 0:
        axes[idx].set_ylabel("")

fig.suptitle("Per-Component Within vs Between-Band Jaccard", fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, "deep_01_component_jaccard.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_01_component_jaccard.png


### 1b. Per-Component Universal/Band-Specific Fractions

For edges at each sharing level (1-5 bands), what is the component type breakdown? Tests H6: are band-specific edges disproportionately MLP?

In [5]:
# Compute edge sharing with properties for all models
all_sharing_dfs = []

for model in MODELS:
    for draw in DRAWS:
        df_sharing = compute_edge_sharing_with_properties(circuits, model, draw)
        if not df_sharing.empty:
            df_sharing["model"] = model
            df_sharing["draw"] = draw
            all_sharing_dfs.append(df_sharing)

df_all_sharing = pd.concat(all_sharing_dfs, ignore_index=True)
print(f"Total edge records: {len(df_all_sharing)}")
print(f"\nSharing level distribution:")
print(df_all_sharing.groupby(["model", "sharing_level"]).size().unstack(fill_value=0))

Total edge records: 50735

Sharing level distribution:
sharing_level     1     2     3     4     5
model                                      
pythia-1.4b    6947  3158  1918  1459  1860
pythia-160m    1892  1223   888   849  2140
pythia-1b      1777  1005   768   573  1068
pythia-410m    8376  4308  2894  2126  3908
pythia-70m      231   162   141   159   905


In [6]:
# Cross-tabulate: sharing level x component type
dst_type_simple = {"attn_in": "attn", "mlp_in": "mlp", "resid_post": "resid"}
df_all_sharing["component"] = df_all_sharing["dst_type"].map(dst_type_simple)

sharing_component_rows = []
for model in MODELS:
    sub = df_all_sharing[df_all_sharing["model"] == model]
    for level in sorted(sub["sharing_level"].unique()):
        level_sub = sub[sub["sharing_level"] == level]
        total = len(level_sub)
        for comp in ["attn", "mlp", "resid"]:
            count = (level_sub["component"] == comp).sum()
            sharing_component_rows.append(
                {
                    "model": model,
                    "sharing_level": level,
                    "component": comp,
                    "count": count,
                    "fraction": count / total if total > 0 else 0,
                }
            )

df_sharing_by_comp = pd.DataFrame(sharing_component_rows)
df_sharing_by_comp.to_csv(ANALYSIS_DIR / "deep_sharing_by_component.csv", index=False)
print(f"Saved: {ANALYSIS_DIR / 'deep_sharing_by_component.csv'}")

# Display pivot: sharing level (rows) x component (cols) for each model
for model in MODELS:
    print(f"\n{model}:")
    sub = df_sharing_by_comp[df_sharing_by_comp["model"] == model]
    pivot = sub.pivot(index="sharing_level", columns="component", values="fraction")
    print(pivot.round(3).to_string())

Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/analysis/deep_sharing_by_component.csv

pythia-70m:
component       attn    mlp  resid
sharing_level                     
1              0.931  0.052  0.017
2              0.889  0.111  0.000
3              0.872  0.099  0.028
4              0.830  0.164  0.006
5              0.467  0.367  0.166

pythia-160m:
component       attn    mlp  resid
sharing_level                     
1              0.829  0.163  0.008
2              0.778  0.214  0.007
3              0.727  0.261  0.011
4              0.677  0.292  0.031
5              0.389  0.451  0.160

pythia-410m:
component       attn    mlp  resid
sharing_level                     
1              0.753  0.237  0.010
2              0.695  0.291  0.014
3              0.680  0.302  0.018
4              0.622  0.339  0.039
5              0.329  0.502  0.168

pythia-1b:
component       attn    mlp  resid
sharing_level                     
1              0.756  0.226  0.017
2           

In [7]:
# Visualization: Stacked bar of component breakdown by sharing level
fig, axes = plt.subplots(1, len(MODELS), figsize=(5 * len(MODELS), 5), sharey=True)

for idx, model in enumerate(MODELS):
    ax = axes[idx]
    sub = df_sharing_by_comp[df_sharing_by_comp["model"] == model]
    pivot = sub.pivot(index="sharing_level", columns="component", values="fraction")
    pivot = pivot[["attn", "mlp", "resid"]]  # consistent order

    pivot.plot(
        kind="bar",
        stacked=True,
        ax=ax,
        color=[
            COMPONENT_COLORS["attn"],
            COMPONENT_COLORS["mlp"],
            COMPONENT_COLORS["resid"],
        ],
    )
    ax.set_xlabel("Sharing Level (# bands)")
    ax.set_title(model)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    if idx == 0:
        ax.set_ylabel("Fraction")
    else:
        ax.get_legend().remove()

fig.suptitle("Component Type Breakdown by Edge Sharing Level", fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, "deep_02_sharing_by_component.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_02_sharing_by_component.png


### 1c. Component Interaction Wiring

Do cross-component wiring patterns (attn->mlp, mlp->resid, etc.) change across bands?

In [8]:
# Extract component flow from each circuit (already pre-computed in JSON)
wiring_rows = []
flow_types = [
    "embed_to_attn",
    "embed_to_mlp",
    "embed_to_resid",
    "attn_to_attn",
    "attn_to_mlp",
    "attn_to_resid",
    "mlp_to_attn",
    "mlp_to_mlp",
    "mlp_to_resid",
]

for c in circuits.values():
    flow = c.get("component_flow", {}).get("flow", {})
    total = c["total_edges"]
    row = {
        "model": c["model"],
        "band": c["band"],
        "draw": c["draw"],
        "total_edges": total,
    }

    for ft in flow_types:
        src, _, dst = ft.partition("_to_")
        count = flow.get(src, {}).get(dst, 0)
        row[f"{ft}_count"] = count
        row[f"{ft}_frac"] = count / total if total > 0 else 0

    wiring_rows.append(row)

df_wiring = pd.DataFrame(wiring_rows)
df_wiring.to_csv(ANALYSIS_DIR / "deep_component_wiring.csv", index=False)
print(f"Saved: {ANALYSIS_DIR / 'deep_component_wiring.csv'}")

# Summary: mean wiring fractions by model x band
frac_cols = [c for c in df_wiring.columns if c.endswith("_frac")]
wiring_summary = df_wiring.groupby(["model", "band"])[frac_cols].mean()
for model in MODELS:
    print(f"\n{model}:")
    sub = wiring_summary.loc[model]
    print(sub.round(4).to_string())

Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/analysis/deep_component_wiring.csv

pythia-70m:
           embed_to_attn_frac  embed_to_mlp_frac  embed_to_resid_frac  attn_to_attn_frac  attn_to_mlp_frac  attn_to_resid_frac  mlp_to_attn_frac  mlp_to_mlp_frac  mlp_to_resid_frac
band                                                                                                                                                                                
control                0.0252             0.0134               0.0024             0.3839            0.2440              0.1062            0.1753           0.0354             0.0142
high                   0.0304             0.0128               0.0024             0.3641            0.2448              0.1072            0.1878           0.0360             0.0144
low                    0.0266             0.0146               0.0026             0.3178            0.2691              0.1114            0.2040           0.0386           

In [9]:
# Visualization: Component wiring heatmaps per model, comparing bands
src_types = ["embed", "attn", "mlp"]
dst_types = ["attn", "mlp", "resid"]

fig, axes = plt.subplots(
    len(MODELS), len(BANDS), figsize=(3.5 * len(BANDS), 3 * len(MODELS))
)

for i, model in enumerate(MODELS):
    for j, band in enumerate(BANDS):
        ax = axes[i, j]
        sub = df_wiring[(df_wiring["model"] == model) & (df_wiring["band"] == band)]

        matrix = np.zeros((3, 3))
        for si, src in enumerate(src_types):
            for di, dst in enumerate(dst_types):
                col = f"{src}_to_{dst}_frac"
                matrix[si, di] = sub[col].mean()

        sns.heatmap(
            matrix,
            ax=ax,
            cmap="Blues",
            vmin=0,
            xticklabels=dst_types if i == len(MODELS) - 1 else [],
            yticklabels=src_types if j == 0 else [],
            annot=True,
            fmt=".3f",
            square=True,
            linewidths=0,
            linecolor="none",
            cbar=False,
        )
        if i == 0:
            ax.set_title(BAND_NAMES.get(band, band), fontsize=10)
        if j == 0:
            ax.set_ylabel(model, fontsize=10)

fig.suptitle(
    "Component Interaction Wiring (fraction of total edges)", fontsize=14, y=1.01
)
fig.tight_layout()
save_figure(fig, "deep_03_component_wiring.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_03_component_wiring.png


---
## 2. Universal vs. Band-Specific Edge Deep Characterization

The existing analysis counts universal/band-specific edges but doesn't characterize **what those edges look like**.

### 2a. Structural Profile by Sharing Level

For each sharing group (1-band-unique through 5-band-universal): where do they concentrate in layer, component, and edge category?

In [10]:
# Aggregate sharing profiles across draws
sharing_profile_rows = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    sub = df_all_sharing[df_all_sharing["model"] == model]

    for level in sorted(sub["sharing_level"].unique()):
        level_sub = sub[sub["sharing_level"] == level]
        total = len(level_sub)

        sharing_profile_rows.append(
            {
                "model": model,
                "sharing_level": level,
                "n_edges": total,
                "mean_layer_distance": level_sub["layer_distance"].mean(),
                "mean_dst_layer": level_sub["dst_layer"].mean(),
                "mean_dst_layer_normalized": level_sub["dst_layer"].mean()
                / (n_layers - 1),
                "skip_fraction": level_sub["is_skip"].mean(),
                "input_fraction": level_sub["is_input"].mean(),
                "output_fraction": level_sub["is_output"].mean(),
                "attn_fraction": (level_sub["component"] == "attn").mean(),
                "mlp_fraction": (level_sub["component"] == "mlp").mean(),
                "resid_fraction": (level_sub["component"] == "resid").mean(),
            }
        )

df_sharing_profiles = pd.DataFrame(sharing_profile_rows)
df_sharing_profiles.to_csv(ANALYSIS_DIR / "deep_sharing_profiles.csv", index=False)
print(f"Saved: {ANALYSIS_DIR / 'deep_sharing_profiles.csv'}")
print(df_sharing_profiles.to_string(index=False))

Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/analysis/deep_sharing_profiles.csv
      model  sharing_level  n_edges  mean_layer_distance  mean_dst_layer  mean_dst_layer_normalized  skip_fraction  input_fraction  output_fraction  attn_fraction  mlp_fraction  resid_fraction
 pythia-70m              1      231             2.047619        3.203463                   0.640693       0.632035        0.043290         0.017316       0.930736      0.051948        0.017316
 pythia-70m              2      162             2.055556        3.296296                   0.659259       0.617284        0.018519         0.000000       0.888889      0.111111        0.000000
 pythia-70m              3      141             2.000000        3.390071                   0.678014       0.617021        0.021277         0.028369       0.872340      0.099291        0.028369
 pythia-70m              4      159             2.106918        3.301887                   0.660377       0.641509        0.012579       

In [11]:
# Visualization: Sharing profile multi-panel
metrics = [
    "mean_dst_layer_normalized",
    "skip_fraction",
    "attn_fraction",
    "mlp_fraction",
    "mean_layer_distance",
]
metric_labels = [
    "Mean Dest Layer (norm)",
    "Skip Fraction",
    "Attention Fraction",
    "MLP Fraction",
    "Mean Layer Distance",
]

fig, axes = plt.subplots(1, len(metrics), figsize=(4 * len(metrics), 4))

for i, (metric, label) in enumerate(zip(metrics, metric_labels)):
    ax = axes[i]
    for model in MODELS:
        sub = df_sharing_profiles[df_sharing_profiles["model"] == model]
        ax.plot(
            sub["sharing_level"],
            sub[metric],
            "o-",
            label=model,
            color=MODEL_COLORS.get(model),
            markersize=6,
        )
    ax.set_xlabel("Sharing Level (# bands)")
    ax.set_ylabel(label)
    ax.set_xticks([1, 2, 3, 4, 5])
    if i == 0:
        ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle("Edge Properties by Sharing Level", fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, "deep_04_sharing_profiles.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_04_sharing_profiles.png


### 2b. Band Affinity Matrix

Among non-universal edges, which band pairs share the most? Does edge sharing follow frequency distance? (Tests H7)

In [12]:
# Compute band affinity summaries
all_affinity_rows = []

for model in MODELS:
    df_aff = compute_band_affinity_summary(circuits, model)
    all_affinity_rows.append(df_aff)

df_all_affinity = pd.concat(all_affinity_rows, ignore_index=True)
df_all_affinity.to_csv(ANALYSIS_DIR / "deep_band_affinity.csv", index=False)
print(f"Saved: {ANALYSIS_DIR / 'deep_band_affinity.csv'}")
print(df_all_affinity.to_string(index=False))

Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/analysis/deep_band_affinity.csv
      model    band_1    band_2  affinity  freq_distance
 pythia-70m       low    medium  0.318648            1.0
 pythia-70m       low      high  0.300848            2.0
 pythia-70m       low very_high  0.214191            3.0
 pythia-70m       low   control  0.255175            NaN
 pythia-70m    medium      high  0.349568            1.0
 pythia-70m    medium very_high  0.267203            2.0
 pythia-70m    medium   control  0.273303            NaN
 pythia-70m      high very_high  0.379080            1.0
 pythia-70m      high   control  0.374640            NaN
 pythia-70m very_high   control  0.395983            NaN
pythia-160m       low    medium  0.372530            1.0
pythia-160m       low      high  0.289677            2.0
pythia-160m       low very_high  0.221999            3.0
pythia-160m       low   control  0.281340            NaN
pythia-160m    medium      high  0.306532            1.0


In [13]:
# Visualization: Affinity heatmaps
fig, axes = plt.subplots(1, len(MODELS), figsize=(5 * len(MODELS), 4.5))

for idx, model in enumerate(MODELS):
    # Average affinity matrix across draws
    matrices = []
    for draw in DRAWS:
        mat, labels = compute_band_affinity_matrix(circuits, model, draw)
        matrices.append(mat)
    avg_mat = np.mean(matrices, axis=0)

    plot_band_affinity_heatmap(avg_mat, labels, model, ax=axes[idx])

fig.suptitle("Band Affinity (non-universal edge Jaccard)", fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, "deep_05_band_affinity.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_05_band_affinity.png


In [14]:
# Scatter: affinity vs frequency distance (tests H7)
fig, axes = plt.subplots(1, len(MODELS), figsize=(4.5 * len(MODELS), 4), sharey=True)

for idx, model in enumerate(MODELS):
    ax = axes[idx]
    sub = df_all_affinity[
        (df_all_affinity["model"] == model) & df_all_affinity["freq_distance"].notna()
    ]

    if not sub.empty:
        ax.scatter(
            sub["freq_distance"],
            sub["affinity"],
            s=60,
            alpha=0.7,
            color=MODEL_COLORS.get(model),
        )

        # Fit line
        from scipy import stats as sp_stats

        rho, p = sp_stats.spearmanr(sub["freq_distance"], sub["affinity"])
        z = np.polyfit(sub["freq_distance"], sub["affinity"], 1)
        px = np.linspace(sub["freq_distance"].min(), sub["freq_distance"].max(), 50)
        ax.plot(px, np.polyval(z, px), "--", color="gray", alpha=0.7)
        ax.text(
            0.05,
            0.95,
            f"ρ={rho:.3f}\np={p:.4f}",
            transform=ax.transAxes,
            va="top",
            fontsize=9,
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5),
        )

    ax.set_xlabel("Frequency Distance")
    ax.set_title(model)
    if idx == 0:
        ax.set_ylabel("Non-Universal Edge Affinity")
    ax.grid(True, alpha=0.3)

fig.suptitle("Structural Affinity vs Frequency Distance (H7)", fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, "deep_06_affinity_vs_distance.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_06_affinity_vs_distance.png


### 2c. Band-Specific Edge Signatures

For edges unique to each band (sharing_level=1): layer distribution, component type, heads involved.

In [15]:
# Extract band-unique edge signatures
band_unique = df_all_sharing[df_all_sharing["sharing_level"] == 1].copy()

sig_rows = []
for model in MODELS:
    sub = band_unique[band_unique["model"] == model]
    n_layers = MODEL_INFO[model]["n_layers"]

    for band in BANDS:
        band_sub = sub[sub["bands_present"] == band]
        if band_sub.empty:
            continue

        sig_rows.append(
            {
                "model": model,
                "band": band,
                "n_unique_edges": len(band_sub),
                "mean_dst_layer": band_sub["dst_layer"].mean(),
                "mean_dst_layer_norm": band_sub["dst_layer"].mean() / (n_layers - 1),
                "attn_frac": (band_sub["component"] == "attn").mean(),
                "mlp_frac": (band_sub["component"] == "mlp").mean(),
                "resid_frac": (band_sub["component"] == "resid").mean(),
                "skip_frac": band_sub["is_skip"].mean(),
                "mean_layer_dist": band_sub["layer_distance"].mean(),
            }
        )

df_signatures = pd.DataFrame(sig_rows)
df_signatures.to_csv(ANALYSIS_DIR / "deep_band_signatures.csv", index=False)
print(f"Saved: {ANALYSIS_DIR / 'deep_band_signatures.csv'}")
print(df_signatures.to_string(index=False))

Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/analysis/deep_band_signatures.csv
      model      band  n_unique_edges  mean_dst_layer  mean_dst_layer_norm  attn_frac  mlp_frac  resid_frac  skip_frac  mean_layer_dist
 pythia-70m       low              35        3.857143             0.771429   0.885714  0.085714    0.028571   0.542857         1.914286
 pythia-70m    medium              29        3.206897             0.641379   0.896552  0.068966    0.034483   0.827586         2.206897
 pythia-70m      high              41        3.121951             0.624390   1.000000  0.000000    0.000000   0.658537         2.243902
 pythia-70m very_high              64        3.031250             0.606250   0.921875  0.062500    0.015625   0.609375         1.937500
 pythia-70m   control              62        3.064516             0.612903   0.935484  0.048387    0.016129   0.596774         2.032258
pythia-160m       low             384        7.080729             0.643703   0.809896  0.17968

In [16]:
# Visualization: Layer distribution of band-unique edges
fig, axes = plt.subplots(1, len(MODELS), figsize=(5 * len(MODELS), 4), sharey=False)

for idx, model in enumerate(MODELS):
    ax = axes[idx]
    sub = band_unique[band_unique["model"] == model]
    n_layers = MODEL_INFO[model]["n_layers"]

    for band in BANDS:
        band_sub = sub[sub["bands_present"] == band]
        if band_sub.empty:
            continue
        counts, bins = np.histogram(band_sub["dst_layer"], bins=range(n_layers + 1))
        ax.plot(
            range(n_layers),
            counts,
            "o-",
            label=BAND_NAMES.get(band, band),
            color=BAND_COLORS.get(band),
            markersize=4,
            alpha=0.8,
        )

    ax.set_xlabel("Destination Layer")
    ax.set_title(model)
    if idx == 0:
        ax.set_ylabel("# Band-Unique Edges")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle("Layer Distribution of Band-Unique Edges", fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, "deep_07_band_unique_layer_dist.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_07_band_unique_layer_dist.png


### 2d. Directed Containment Analysis

Does low-frequency circuit contain most of high-frequency circuit's edges? (Tests H5: subset hypothesis)

In [17]:
# Compute directed containment
all_containment_dfs = []
for model in MODELS:
    df_cont = compute_directed_containment(circuits, model)
    all_containment_dfs.append(df_cont)

df_containment = pd.concat(all_containment_dfs, ignore_index=True)
df_containment.to_csv(ANALYSIS_DIR / "deep_directed_containment.csv", index=False)
print(f"Saved: {ANALYSIS_DIR / 'deep_directed_containment.csv'}")

# Summary: mean containment across draws
cont_summary = (
    df_containment.groupby(["model", "source_band", "target_band"])["containment"]
    .mean()
    .reset_index()
)
for model in MODELS:
    print(
        f"\n{model}: Containment(source contains target): fraction of target's edges found in source"
    )
    sub = cont_summary[cont_summary["model"] == model]
    pivot = sub.pivot(index="source_band", columns="target_band", values="containment")
    pivot = pivot.reindex(index=BANDS, columns=BANDS)
    print(pivot.round(3).to_string())

# Edge count context: containment interpretation depends on relative edge counts.
# Containment(A contains B) = |A  and  B| / |B|. When A has more edges than B, this is
# mechanically higher because A covers more of the edge space. H5 predicts
# low-freq circuits contain high-freq circuits (low has more edges), but this only
# holds when low-freq bands actually have more edges.
print("\n--- Edge count ordering (mean across draws) ---")
size_summary = df_containment.groupby(["model", "source_band"])["source_size"].mean()
for model in MODELS:
    sizes = {b: size_summary.get((model, b), 0) for b in BANDS}
    ordered = sorted(sizes.items(), key=lambda x: x[1], reverse=True)
    ordering = " > ".join(f"{b}({int(s)})" for b, s in ordered)
    # Check if frequency ordering matches size ordering
    freq_bands_only = {b: s for b, s in sizes.items() if b != "control"}
    low_bigger = freq_bands_only.get("low", 0) > freq_bands_only.get("very_high", 0)
    note = (
        "low > very_high (matches H5 direction)"
        if low_bigger
        else "very_high > low (opposite of H5)"
    )
    print(f"  {model}: {ordering}")
    print(f"    -> {note}")

Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/analysis/deep_directed_containment.csv

pythia-70m: Containment(source contains target): fraction of target's edges found in source
target_band    low  medium   high  very_high  control
source_band                                          
low            NaN   0.870  0.836      0.806    0.813
medium       0.889     NaN  0.855      0.827    0.823
high         0.896   0.896    NaN      0.872    0.865
very_high    0.869   0.872  0.878        NaN    0.873
control      0.886   0.876  0.879      0.882      NaN

pythia-160m: Containment(source contains target): fraction of target's edges found in source
target_band    low  medium   high  very_high  control
source_band                                          
low            NaN   0.758  0.737      0.721    0.744
medium       0.772     NaN  0.752      0.737    0.745
high         0.711   0.712    NaN      0.731    0.720
very_high    0.662   0.665  0.696        NaN    0.712
control      0.7

In [18]:
# Visualization: Containment heatmaps
fig, axes = plt.subplots(1, len(MODELS), figsize=(5 * len(MODELS), 4.5))

for idx, model in enumerate(MODELS):
    ax = axes[idx]
    sub = cont_summary[cont_summary["model"] == model]
    pivot = sub.pivot(index="source_band", columns="target_band", values="containment")
    pivot = pivot.reindex(index=BANDS, columns=BANDS)

    display_bands = [BAND_NAMES.get(b, b) for b in BANDS]
    sns.heatmap(
        pivot.values,
        ax=ax,
        cmap="YlOrRd",
        xticklabels=display_bands,
        yticklabels=display_bands,
        annot=True,
        fmt=".2f",
        square=True,
        linewidths=0,
        linecolor="none",
        cbar_kws={"label": "Containment"},
    )
    ax.set_xlabel("Target Band")
    ax.set_ylabel("Source Band")
    ax.set_title(model)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

fig.suptitle(
    "Directed Containment: fraction of target edges found in source (H5)",
    fontsize=14,
    y=1.02,
)
fig.tight_layout()
save_figure(fig, "deep_08_directed_containment.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_08_directed_containment.png


---
## 3. Layer-Level Decomposition

### 3a. Layer-wise Band Sensitivity Profile

For each layer, compute mean pairwise Jaccard between bands. Low Jaccard = high sensitivity. (Tests H9: contiguous sensitive zone)

In [19]:
# Compute layer sensitivity for all models
all_layer_sens = []
for model in MODELS:
    print(f"Computing layer sensitivity for {model}...")
    df_sens = compute_layer_band_sensitivity(circuits, model)
    all_layer_sens.append(df_sens)

df_layer_sensitivity = pd.concat(all_layer_sens, ignore_index=True)
df_layer_sensitivity.to_csv(ANALYSIS_DIR / "deep_layer_sensitivity.csv", index=False)
print(f"\nSaved: {ANALYSIS_DIR / 'deep_layer_sensitivity.csv'}")
print(df_layer_sensitivity.to_string(index=False))

Computing layer sensitivity for pythia-70m...
Computing layer sensitivity for pythia-160m...
Computing layer sensitivity for pythia-410m...


Computing layer sensitivity for pythia-1b...
Computing layer sensitivity for pythia-1.4b...



Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/analysis/deep_layer_sensitivity.csv
      model  layer  mean_jaccard  std_jaccard  n_pairs
 pythia-70m      0      1.000000     0.000000       90
 pythia-70m      1      0.763368     0.059454       90
 pythia-70m      2      0.656607     0.052514       90
 pythia-70m      3      0.785459     0.037635       90
 pythia-70m      4      0.662656     0.072027       90
 pythia-70m      5      0.839166     0.033523       90
pythia-160m      0      0.785606     0.099963       90
pythia-160m      1      0.506051     0.071268       90
pythia-160m      2      0.464376     0.060562       90
pythia-160m      3      0.422665     0.051573       90
pythia-160m      4      0.515326     0.043745       90
pythia-160m      5      0.578030     0.035938       90
pythia-160m      6      0.456979     0.031034       90
pythia-160m      7      0.494606     0.031587       90
pythia-160m      8      0.538951     0.041457       90
pythia-160m      9      0.5

In [20]:
# Visualization: Layer sensitivity profile
fig, ax = plt.subplots(figsize=(14, 6))
plot_layer_sensitivity_profile(df_layer_sensitivity, ax=ax)
save_figure(fig, "deep_09_layer_sensitivity.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_09_layer_sensitivity.png


### 3b. Per-Layer Universal Fraction

In [21]:
# Compute per-layer universal fraction
all_layer_univ = []
for model in MODELS:
    for draw in DRAWS:
        df_lu = compute_per_layer_universal_fraction(circuits, model, draw)
        all_layer_univ.append(df_lu)

df_layer_universal = pd.concat(all_layer_univ, ignore_index=True)
df_layer_universal.to_csv(
    ANALYSIS_DIR / "deep_layer_universal_fraction.csv", index=False
)
print(f"Saved: {ANALYSIS_DIR / 'deep_layer_universal_fraction.csv'}")

Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/analysis/deep_layer_universal_fraction.csv


In [22]:
# Visualization: Per-layer universal fraction
fig, ax = plt.subplots(figsize=(14, 6))

for model in MODELS:
    sub = df_layer_universal[df_layer_universal["model"] == model]
    avg = sub.groupby("layer")["universal_fraction"].agg(["mean", "std"]).reset_index()
    color = MODEL_COLORS.get(model)
    ax.plot(
        avg["layer"],
        avg["mean"],
        "o-",
        label=model,
        color=color,
        markersize=5,
        linewidth=2,
    )
    ax.fill_between(
        avg["layer"],
        avg["mean"] - avg["std"],
        avg["mean"] + avg["std"],
        alpha=0.15,
        color=color,
    )

ax.set_xlabel("Destination Layer")
ax.set_ylabel("Universal Edge Fraction")
ax.set_title("Per-Layer Universal Edge Fraction")
ax.legend()
ax.grid(True, alpha=0.3)
save_figure(fig, "deep_10_layer_universal_fraction.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_10_layer_universal_fraction.png


### 3c. Layer Flow Difference Maps

Compute pairwise differences in layer-to-layer flow matrices between bands.

In [23]:
# Compute mean layer flow per model x band
def get_flow_matrix(circuit, n_layers):
    """Convert circuit flow dict to numpy matrix."""
    src = n_layers + 1  # embed + layers
    dst = n_layers
    mat = np.zeros((src, dst))
    for s_str, d_dict in circuit.get("layer_flow", {}).get("flow", {}).items():
        s = int(s_str) + 1
        if 0 <= s < src:
            for d_str, cnt in d_dict.items():
                d = int(d_str)
                if 0 <= d < dst:
                    mat[s, d] = cnt
    return mat


# Compute and save difference maps for key pairs: one figure per model
diff_pairs = [("low", "very_high"), ("low", "high"), ("medium", "very_high")]

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    n_heads = MODEL_INFO[model]["n_heads"]

    # Gather per-band flow matrices
    band_flows = {}
    for band in BANDS:
        mats = []
        for draw in DRAWS:
            key = f"{model.replace('-', '_')}_{band}_{draw}"
            if key in circuits:
                mats.append(get_flow_matrix(circuits[key], n_layers))
        if mats:
            band_flows[band] = np.mean(mats, axis=0)

    fig, axes = plt.subplots(
        1, len(diff_pairs), figsize=(5 * len(diff_pairs), max(4, (n_layers + 1) * 0.35))
    )

    for j, (b1, b2) in enumerate(diff_pairs):
        ax = axes[j]
        if b1 in band_flows and b2 in band_flows:
            diff = band_flows[b1] - band_flows[b2]

            src_labels = ["emb"] + [f"L{l}" for l in range(n_layers)]
            dst_labels = [f"L{l}" for l in range(n_layers)]

            vmax = max(abs(diff.min()), abs(diff.max()))
            if vmax == 0:
                vmax = 1
            sns.heatmap(
                diff,
                ax=ax,
                cmap="RdBu_r",
                center=0,
                vmin=-vmax,
                vmax=vmax,
                xticklabels=dst_labels,
                yticklabels=src_labels if j == 0 else [],
                square=True,
                linewidths=0,
                linecolor="none",
                cbar=True,
                cbar_kws={"shrink": 0.6},
            )
        ax.set_title(
            f"{b1.replace('_', ' ').title()} \u2212 {b2.replace('_', ' ').title()}"
        )
        if j == 0:
            ax.set_ylabel("Source")
        ax.set_xlabel("Destination")

    fig.suptitle(f"Layer Flow Difference Maps \u2014 {model}", fontsize=14, y=1.01)
    fig.tight_layout()
    save_figure(fig, f"deep_11_layer_flow_diffs_{model}.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_11_layer_flow_diffs_pythia-70m.png


  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_11_layer_flow_diffs_pythia-160m.png


  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_11_layer_flow_diffs_pythia-410m.png


  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_11_layer_flow_diffs_pythia-1b.png


  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_11_layer_flow_diffs_pythia-1.4b.png


---
## 4. Head-Level Decomposition

### 4a. Head Universality Scores

For each head: in how many bands is it active?

In [24]:
# Compute head band presence
all_head_presence = []
for model in MODELS:
    df_hp = compute_head_band_presence(circuits, model)
    all_head_presence.append(df_hp)

df_head_presence = pd.concat(all_head_presence, ignore_index=True)
df_head_presence.to_csv(ANALYSIS_DIR / "deep_head_universality.csv", index=False)
print(f"Saved: {ANALYSIS_DIR / 'deep_head_universality.csv'}")

# Summary: universality distribution per model
for model in MODELS:
    sub = df_head_presence[df_head_presence["model"] == model]
    avg = sub.groupby("head")["n_bands"].mean()
    print(
        f"\n{model}: mean universality={avg.mean():.2f}, "
        f"universal heads (5/5)={(avg == 5).sum()}, "
        f"band-unique (1/5)={(avg <= 1).sum()}, "
        f"inactive (0/5)={(avg == 0).sum()}"
    )

Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/analysis/deep_head_universality.csv

pythia-70m: mean universality=4.25, universal heads (5/5)=31, band-unique (1/5)=4, inactive (0/5)=3

pythia-160m: mean universality=3.41, universal heads (5/5)=59, band-unique (1/5)=27, inactive (0/5)=17

pythia-410m: mean universality=2.79, universal heads (5/5)=125, band-unique (1/5)=128, inactive (0/5)=79

pythia-1b: mean universality=3.10, universal heads (5/5)=47, band-unique (1/5)=32, inactive (0/5)=15

pythia-1.4b: mean universality=2.21, universal heads (5/5)=79, band-unique (1/5)=169, inactive (0/5)=100


In [25]:
# Visualization: Head universality heatmaps: separate per model
for model in MODELS:
    info = MODEL_INFO[model]
    sub = df_head_presence[df_head_presence["model"] == model]
    fig, ax = plt.subplots(
        figsize=(max(6, info["n_layers"] * 0.55), max(4, info["n_heads"] * 0.45))
    )
    plot_head_universality_map(
        sub,
        model,
        info["n_layers"],
        info["n_heads"],
        title=f"Head Universality: {model}",
        ax=ax,
    )
    fig.tight_layout()
    save_figure(fig, f"deep_12_head_universality_{model}.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_12_head_universality_pythia-70m.png


  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_12_head_universality_pythia-160m.png


  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_12_head_universality_pythia-410m.png


  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_12_head_universality_pythia-1b.png


  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_12_head_universality_pythia-1.4b.png


### 4b. Band-Discriminative Head Identification

Entropy of each head's edge-count distribution across bands. Low entropy = band-discriminative.

In [26]:
# Compute head band entropy
all_head_entropy = []
for model in MODELS:
    df_he = compute_head_band_entropy(circuits, model)
    all_head_entropy.append(df_he)

df_head_entropy = pd.concat(all_head_entropy, ignore_index=True)
df_head_entropy.to_csv(ANALYSIS_DIR / "deep_head_entropy.csv", index=False)
print(f"Saved: {ANALYSIS_DIR / 'deep_head_entropy.csv'}")

# Top 10 most discriminative heads per model (lowest normalized entropy, excluding inactive)
for model in MODELS:
    sub = df_head_entropy[
        (df_head_entropy["model"] == model) & (df_head_entropy["total_edges"] > 0)
    ]
    avg = (
        sub.groupby("head")
        .agg(
            {
                "normalized_entropy": "mean",
                "dominant_band": lambda x: (
                    x.mode().iloc[0] if len(x.mode()) > 0 else None
                ),
                "total_edges": "mean",
            }
        )
        .reset_index()
    )
    avg = avg.sort_values("normalized_entropy")
    print(f"\n{model}: Top 10 most discriminative heads:")
    print(avg.head(10).to_string(index=False))

Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/analysis/deep_head_entropy.csv

pythia-70m: Top 10 most discriminative heads:
head  normalized_entropy dominant_band  total_edges
A4.3            0.000000       control     1.000000
A1.0            0.483930           low     3.000000
A4.1            0.535935     very_high     5.000000
A1.7            0.646483          high     3.333333
A4.5            0.677764           low     6.000000
A5.5            0.680012     very_high     8.333333
A4.6            0.712381           low    14.333333
A3.2            0.736369           low     7.333333
A4.4            0.772664           low    21.333333
A1.5            0.847986           low     4.000000



pythia-160m: Top 10 most discriminative heads:
  head  normalized_entropy dominant_band  total_edges
  A7.5            0.000000     very_high     1.000000
  A2.5            0.000000           low     1.000000
  A8.9            0.000000     very_high     2.000000
 A3.11            0.000000       control     1.000000
A11.10            0.000000           low     3.500000
  A3.5            0.000000       control     1.666667
  A7.1            0.000000          high     1.500000
  A7.9            0.000000     very_high     1.000000
 A1.11            0.143559       control     1.333333
  A2.0            0.143559           low     1.333333

pythia-410m: Top 10 most discriminative heads:
 head  normalized_entropy dominant_band  total_edges
A4.13                 0.0           low          1.5
A21.9                 0.0        medium          1.0
A21.8                 0.0        medium          2.0
A20.4                 0.0           low          3.0
 A2.7                 0.0       control      


pythia-1b: Top 10 most discriminative heads:
 head  normalized_entropy dominant_band  total_edges
 A3.4                 0.0          high     1.000000
A14.3                 0.0     very_high     1.000000
 A3.0                 0.0          high     1.333333
A12.7                 0.0       control     3.000000
 A0.6                 0.0       control     1.000000
A13.7                 0.0       control     1.000000
A15.4                 0.0           low     1.000000
A14.4                 0.0           low     1.000000
 A2.4                 0.0          high     1.000000
A11.1                 0.0       control     2.000000



pythia-1.4b: Top 10 most discriminative heads:
 head  normalized_entropy dominant_band  total_edges
A20.8                 0.0        medium    19.000000
A2.12                 0.0     very_high     1.000000
A2.14                 0.0        medium     4.000000
 A4.2                 0.0        medium     1.000000
A20.0                 0.0        medium    10.500000
A4.11                 0.0           low     1.666667
A4.10                 0.0           low     1.000000
 A4.0                 0.0       control     1.000000
 A3.9                 0.0           low     1.000000
A10.6                 0.0        medium     1.000000


In [27]:
# Visualization: Head band entropy: separate heatmap per model
# Shows normalized entropy as color in a Head (Y) x Layer (X) grid.
# Low entropy (dark) = discriminative heads; high entropy (light) = uniform.
for model in MODELS:
    info = MODEL_INFO[model]
    n_layers, n_heads = info["n_layers"], info["n_heads"]
    sub = df_head_entropy[df_head_entropy["model"] == model]

    # Build head x layer matrices for entropy and dominant band
    entropy_matrix = np.full((n_heads, n_layers), np.nan)
    dom_matrix = np.empty((n_heads, n_layers), dtype=object)
    dom_matrix[:] = ""

    # Average across draws per head
    for head_name, grp in sub.groupby("head"):
        # Parse head name like 'A3.5' -> layer=3, head_idx=5
        parts = head_name[1:].split(".")
        layer, head_idx = int(parts[0]), int(parts[1])
        if layer < n_layers and head_idx < n_heads:
            avg_ent = grp["normalized_entropy"].mean()
            entropy_matrix[head_idx, layer] = avg_ent
            dom = grp["dominant_band"].mode()
            dom_matrix[head_idx, layer] = (
                dom.iloc[0][0].upper() if len(dom) > 0 and pd.notna(dom.iloc[0]) else ""
            )

    fig, ax = plt.subplots(figsize=(max(6, n_layers * 0.55), max(4, n_heads * 0.45)))

    # Use a reversed colormap so low entropy (discriminative) is dark
    sns.heatmap(
        entropy_matrix,
        ax=ax,
        cmap="YlOrRd",
        vmin=0,
        vmax=1,
        xticklabels=[f"L{l}" for l in range(n_layers)],
        yticklabels=[f"H{h}" for h in range(n_heads)],
        square=True,
        linewidths=0,
        linecolor="none",
        cbar_kws={"label": "Normalized Entropy"},
        mask=np.isnan(entropy_matrix),
    )

    ax.set_xlabel("Layer")
    ax.set_ylabel("Head")
    ax.set_title(f"Head Band Entropy: {model} (low=discriminative)")
    fig.tight_layout()
    save_figure(fig, f"deep_13_head_entropy_{model}.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_13_head_entropy_pythia-70m.png


  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_13_head_entropy_pythia-160m.png


  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_13_head_entropy_pythia-410m.png


  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_13_head_entropy_pythia-1b.png


  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_13_head_entropy_pythia-1.4b.png


### 4c. Head Connectivity Pattern Clustering

Cluster heads by their connectivity profile (which sources feed them).

In [28]:
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import pdist

cluster_rows = []

for model in MODELS:
    info = MODEL_INFO[model]
    n_layers, n_heads = info["n_layers"], info["n_heads"]

    # Build connectivity profiles: for each head, count edges from each source
    # Sources: embed + (n_heads + 1) per layer up to head's layer
    # Simplified: vector of incoming edge counts by source layer
    head_profiles = {}

    for draw in DRAWS:
        for band in BANDS:
            c = None
            for cv in circuits.values():
                if cv["model"] == model and cv["band"] == band and cv["draw"] == draw:
                    c = cv
                    break
            if c is None:
                continue

            for edge in c.get("edges", []):
                if edge["dst_type"] != "attn_in" or edge.get("dst_head") is None:
                    continue
                head_key = f"A{edge['dst_layer']}.{edge['dst_head']}"
                profile_key = (head_key, band, draw)
                if profile_key not in head_profiles:
                    head_profiles[profile_key] = np.zeros(
                        n_layers + 1
                    )  # embed + layers
                src_idx = edge["src_layer"] + 1  # -1 -> 0
                if 0 <= src_idx < n_layers + 1:
                    head_profiles[profile_key][src_idx] += 1

    # Average across draws per (head, band)
    avg_profiles = defaultdict(lambda: np.zeros(n_layers + 1))
    avg_counts = defaultdict(int)
    for (head, band, draw), profile in head_profiles.items():
        avg_profiles[(head, band)] += profile
        avg_counts[(head, band)] += 1

    for key in avg_profiles:
        avg_profiles[key] /= max(avg_counts[key], 1)

    # Build matrix: heads x features (concatenate profiles across bands or use mean)
    # Use mean profile across bands for clustering
    all_heads = sorted(set(h for h, b in avg_profiles.keys()))
    head_matrix = []
    head_labels = []

    for head in all_heads:
        profiles = [avg_profiles[(head, b)] for b in BANDS if (head, b) in avg_profiles]
        if profiles:
            mean_profile = np.mean(profiles, axis=0)
            head_matrix.append(mean_profile)
            head_labels.append(head)

    if len(head_matrix) < 3:
        continue

    head_matrix = np.array(head_matrix)

    # Normalize
    row_sums = head_matrix.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    head_matrix_norm = head_matrix / row_sums

    # Hierarchical clustering
    if head_matrix_norm.shape[0] > 1:
        dists = pdist(head_matrix_norm, metric="cosine")
        dists = np.nan_to_num(dists, nan=1.0)
        Z = linkage(dists, method="ward")
        n_clusters = min(5, len(head_labels))
        labels = fcluster(Z, t=n_clusters, criterion="maxclust")

        for head, cluster in zip(head_labels, labels):
            # Get band presence for this head
            presence = df_head_presence[
                (df_head_presence["model"] == model)
                & (df_head_presence["head"] == head)
            ]
            avg_bands = presence["n_bands"].mean() if not presence.empty else 0

            cluster_rows.append(
                {
                    "model": model,
                    "head": head,
                    "cluster": int(cluster),
                    "avg_n_bands": avg_bands,
                }
            )

df_head_clusters = pd.DataFrame(cluster_rows)
df_head_clusters.to_csv(ANALYSIS_DIR / "deep_head_clusters.csv", index=False)
print(f"Saved: {ANALYSIS_DIR / 'deep_head_clusters.csv'}")

# Summary
for model in MODELS:
    sub = df_head_clusters[df_head_clusters["model"] == model]
    if sub.empty:
        continue
    print(f"\n{model}:")
    summary = (
        sub.groupby("cluster")
        .agg(
            n_heads=("head", "count"),
            avg_universality=("avg_n_bands", "mean"),
        )
        .reset_index()
    )
    print(summary.to_string(index=False))

Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/analysis/deep_head_clusters.csv

pythia-70m:
 cluster  n_heads  avg_universality
       1        8          5.000000
       2        2          4.166667
       3       12          4.027778
       4        8          4.791667
       5       15          4.600000

pythia-160m:
 cluster  n_heads  avg_universality
       1       45          3.696296
       2       11          3.757576
       3       27          4.135802
       4       33          4.090909
       5       11          3.333333

pythia-410m:
 cluster  n_heads  avg_universality
       1       73          4.086758
       2       27          2.617284
       3       97          4.302405
       4       14          2.642857
       5       94          2.652482

pythia-1b:
 cluster  n_heads  avg_universality
       1       31          3.419355
       2       31          3.451613
       3        7          2.761905
       4       27          3.925926
       5       17          3.45

---
## 5. Graph-Theoretic Complexity

Do circuits differ in topological complexity (diameter, clustering, hub structure) beyond edge counts?

In [29]:
# Compute graph metrics for all 60 circuits
print("Computing graph metrics for all circuits...")
df_graph = compute_all_graph_metrics(circuits)
df_graph.to_csv(ANALYSIS_DIR / "deep_graph_metrics.csv", index=False)
print(f"Saved: {ANALYSIS_DIR / 'deep_graph_metrics.csv'}")

# Summary by model x band
metric_cols = [
    "diameter",
    "avg_path_length",
    "clustering_coefficient",
    "n_weakly_connected",
    "density",
    "avg_in_degree",
    "max_in_degree",
]
summary = df_graph.groupby(["model", "band"])[metric_cols].mean()
print("\nGraph metrics summary (mean):")
print(summary.round(3).to_string())

Computing graph metrics for all circuits...


Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/analysis/deep_graph_metrics.csv

Graph metrics summary (mean):
                       diameter  avg_path_length  clustering_coefficient  n_weakly_connected  density  avg_in_degree  max_in_degree
model       band                                                                                                                   
pythia-1.4b control       4.333            2.238                   0.539                 1.0    0.035          8.600        160.333
            high          4.000            2.226                   0.533                 1.0    0.036          8.772        157.667
            low           4.000            2.166                   0.580                 1.0    0.038          9.872        173.667
            medium        4.000            2.208                   0.556                 1.0    0.037          9.557        169.000
            very_high     4.000            2.276                   0.507                 

In [30]:
# Visualization: Graph metrics comparison
fig = plot_graph_metrics_comparison(df_graph)
save_figure(fig, "deep_14_graph_metrics.png")

LSC_circuit_analysis/02_Phase_Structural/utils/plotting.py:394: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=sub, x='band', y=metric, order=band_order, ax=ax,
LSC_circuit_analysis/02_Phase_Structural/utils/plotting.py:394: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=sub, x='band', y=metric, order=band_order, ax=ax,
LSC_circuit_analysis/02_Phase_Structural/utils/plotting.py:394: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=sub, x='band', y=metric, order=band_order, ax=ax,


LSC_circuit_analysis/02_Phase_Structural/utils/plotting.py:394: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=sub, x='band', y=metric, order=band_order, ax=ax,


LSC_circuit_analysis/02_Phase_Structural/utils/plotting.py:394: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=sub, x='band', y=metric, order=band_order, ax=ax,
LSC_circuit_analysis/02_Phase_Structural/utils/plotting.py:394: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=sub, x='band', y=metric, order=band_order, ax=ax,
LSC_circuit_analysis/02_Phase_Structural/utils/plotting.py:394: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=sub, x='band', y=metric, order=band_order, ax=ax,


LSC_circuit_analysis/02_Phase_Structural/utils/plotting.py:394: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=sub, x='band', y=metric, order=band_order, ax=ax,
LSC_circuit_analysis/02_Phase_Structural/utils/plotting.py:394: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=sub, x='band', y=metric, order=band_order, ax=ax,
LSC_circuit_analysis/02_Phase_Structural/utils/plotting.py:394: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=sub, x='band', y=metric, order=band_order, ax=ax,


LSC_circuit_analysis/02_Phase_Structural/utils/plotting.py:394: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=sub, x='band', y=metric, order=band_order, ax=ax,
LSC_circuit_analysis/02_Phase_Structural/utils/plotting.py:394: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=sub, x='band', y=metric, order=band_order, ax=ax,
LSC_circuit_analysis/02_Phase_Structural/utils/plotting.py:394: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=sub, x='band', y=metric, order=band_order, ax=ax,


LSC_circuit_analysis/02_Phase_Structural/utils/plotting.py:394: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=sub, x='band', y=metric, order=band_order, ax=ax,
LSC_circuit_analysis/02_Phase_Structural/utils/plotting.py:394: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=sub, x='band', y=metric, order=band_order, ax=ax,
LSC_circuit_analysis/02_Phase_Structural/utils/plotting.py:394: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=sub, x='band', y=metric, order=band_order, ax=ax,


LSC_circuit_analysis/02_Phase_Structural/utils/plotting.py:394: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=sub, x='band', y=metric, order=band_order, ax=ax,
LSC_circuit_analysis/02_Phase_Structural/utils/plotting.py:394: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=sub, x='band', y=metric, order=band_order, ax=ax,
LSC_circuit_analysis/02_Phase_Structural/utils/plotting.py:394: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=sub, x='band', y=metric, order=band_order, ax=ax,


LSC_circuit_analysis/02_Phase_Structural/utils/plotting.py:394: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=sub, x='band', y=metric, order=band_order, ax=ax,


  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_14_graph_metrics.png


In [31]:
# Hub analysis
all_hubs = []
for model in MODELS:
    df_hubs = classify_hub_universality(circuits, model, top_k=10)
    all_hubs.append(df_hubs)

df_all_hubs = pd.concat(all_hubs, ignore_index=True)
df_all_hubs.to_csv(ANALYSIS_DIR / "deep_hub_nodes.csv", index=False)
print(f"Saved: {ANALYSIS_DIR / 'deep_hub_nodes.csv'}")

# Summary: how many hubs are universal?
for model in MODELS:
    sub = df_all_hubs[df_all_hubs["model"] == model]
    for ctype in ["betweenness", "degree"]:
        ct = sub[sub["centrality_type"] == ctype]
        n_universal = ct["is_universal_hub"].sum()
        print(f"{model} ({ctype}): {n_universal}/{len(ct)} hubs are universal")

Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/analysis/deep_hub_nodes.csv
pythia-70m (betweenness): 9/14 hubs are universal
pythia-70m (degree): 10/14 hubs are universal
pythia-160m (betweenness): 10/16 hubs are universal
pythia-160m (degree): 10/15 hubs are universal
pythia-410m (betweenness): 10/21 hubs are universal
pythia-410m (degree): 8/19 hubs are universal
pythia-1b (betweenness): 9/21 hubs are universal
pythia-1b (degree): 11/19 hubs are universal
pythia-1.4b (betweenness): 7/22 hubs are universal
pythia-1.4b (degree): 10/20 hubs are universal


In [32]:
# Degree distribution analysis
all_degrees = []
for c in circuits.values():
    graph = build_circuit_graph(c)
    df_deg = compute_degree_stats(graph)
    df_deg["model"] = c["model"]
    df_deg["band"] = c["band"]
    df_deg["draw"] = c["draw"]
    all_degrees.append(df_deg)

df_all_degrees = pd.concat(all_degrees, ignore_index=True)
df_all_degrees.to_csv(ANALYSIS_DIR / "deep_degree_stats.csv", index=False)
print(f"Saved: {ANALYSIS_DIR / 'deep_degree_stats.csv'}")

Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/analysis/deep_degree_stats.csv


---
## 6. Input/Output Edge Analysis

Do different bands route through different entry points (from embedding) and exit paths (to final residual)?

In [33]:
# Input edges: from embedding
input_rows = []
for c in circuits.values():
    for edge in c.get("edges", []):
        if edge["is_input"]:
            input_rows.append(
                {
                    "model": c["model"],
                    "band": c["band"],
                    "draw": c["draw"],
                    "dst_type": edge["dst_type"],
                    "dst_layer": edge["dst_layer"],
                    "dst_head": edge.get("dst_head"),
                }
            )

df_input = pd.DataFrame(input_rows)
df_input.to_csv(ANALYSIS_DIR / "deep_input_edges.csv", index=False)
print(f"Input edges: {len(df_input)}")

# Output edges: to final resid_post
output_rows = []
for c in circuits.values():
    for edge in c.get("edges", []):
        if edge["is_output"]:
            output_rows.append(
                {
                    "model": c["model"],
                    "band": c["band"],
                    "draw": c["draw"],
                    "src_type": edge["src_type"],
                    "src_layer": edge["src_layer"],
                    "src_head": edge.get("src_head"),
                }
            )

df_output = pd.DataFrame(output_rows)
df_output.to_csv(ANALYSIS_DIR / "deep_output_edges.csv", index=False)
print(f"Output edges: {len(df_output)}")
print(f"Saved: deep_input_edges.csv, deep_output_edges.csv")

Input edges: 961
Output edges: 10089
Saved: deep_input_edges.csv, deep_output_edges.csv


In [34]:
# Visualization: Input edge destinations by band
fig, axes = plt.subplots(1, len(MODELS), figsize=(5 * len(MODELS), 4))

for idx, model in enumerate(MODELS):
    ax = axes[idx]
    sub = df_input[df_input["model"] == model]
    n_layers = MODEL_INFO[model]["n_layers"]

    for band in BANDS:
        band_sub = sub[sub["band"] == band]
        if band_sub.empty:
            continue
        counts = (
            band_sub.groupby("dst_layer").size().reindex(range(n_layers), fill_value=0)
        )
        # Average across draws
        n_draws = band_sub["draw"].nunique()
        counts = counts / max(n_draws, 1)
        ax.plot(
            range(n_layers),
            counts.values,
            "o-",
            label=BAND_NAMES.get(band, band),
            color=BAND_COLORS.get(band),
            markersize=4,
            alpha=0.8,
        )

    ax.set_xlabel("Destination Layer")
    ax.set_title(model)
    if idx == 0:
        ax.set_ylabel("# Input Edges (avg/draw)")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

fig.suptitle("Input Edge Destinations by Band (from embedding)", fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, "deep_15_input_edges.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_15_input_edges.png


In [35]:
# Visualization: Output edge sources by band
fig, axes = plt.subplots(1, len(MODELS), figsize=(5 * len(MODELS), 4))

for idx, model in enumerate(MODELS):
    ax = axes[idx]
    sub = df_output[df_output["model"] == model]
    n_layers = MODEL_INFO[model]["n_layers"]

    for band in BANDS:
        band_sub = sub[sub["band"] == band]
        if band_sub.empty:
            continue
        counts = (
            band_sub.groupby("src_layer").size().reindex(range(n_layers), fill_value=0)
        )
        n_draws = band_sub["draw"].nunique()
        counts = counts / max(n_draws, 1)
        ax.plot(
            range(n_layers),
            counts.values,
            "o-",
            label=BAND_NAMES.get(band, band),
            color=BAND_COLORS.get(band),
            markersize=4,
            alpha=0.8,
        )

    ax.set_xlabel("Source Layer")
    ax.set_title(model)
    if idx == 0:
        ax.set_ylabel("# Output Edges (avg/draw)")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

fig.suptitle("Output Edge Sources by Band (to final residual)", fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, "deep_16_output_edges.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_16_output_edges.png


---
## 7. Draw Stability & Cross-Analyses

Three gap-filling analyses:
- **7a-7c (S-G3)**: Per-edge draw stability: do the same edges appear in all 3 draws?
  Cross-tabulation with band sharing to assess reliability of band-specific edges.
- **7d (S-G4)**: Universal core connectivity: is the universal edge core a connected
  path from embedding to output, or fragmented?
- **7e (S-G5)**: Edge skip distance by sharing level: do band-specific edges have
  systematically different skip distances?

### 7a. Per-Band Draw Stability Distributions

For each model x band, load 3 draws' edge sets and check how many draws each edge appears in (1, 2, or 3). A draw-stability of 3 means the edge is reliably discovered by ACDC across random data samples.

In [36]:
# Compute per-edge draw stability for all model x band combinations
all_stab_dfs = []
for model in MODELS:
    for band in BANDS:
        df_stab = compute_edge_draw_stability(circuits, model, band)
        if not df_stab.empty:
            all_stab_dfs.append(df_stab)

df_draw_stability = pd.concat(all_stab_dfs, ignore_index=True)
df_draw_stability.to_csv(ANALYSIS_DIR / "deep_draw_stability.csv", index=False)
print(f"Saved: {ANALYSIS_DIR / 'deep_draw_stability.csv'}")
print(f"Total edge-band records: {len(df_draw_stability)}")
print(f"\nDraw stability distribution (all models):")
print(df_draw_stability.groupby(["model", "n_draws"]).size().unstack(fill_value=0))

Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/analysis/deep_draw_stability.csv
Total edge-band records: 68610

Draw stability distribution (all models):
n_draws          1     2     3
model                         
pythia-1.4b  10183  4956  4686
pythia-160m   3256  2198  4482
pythia-1b     3038  1819  2349
pythia-410m  13434  7110  8688
pythia-70m     391   312  1708


In [37]:
# Visualization: Per-band draw stability distributions
fig, axes = plt.subplots(
    len(MODELS),
    len(BANDS),
    figsize=(3.5 * len(BANDS), 3 * len(MODELS)),
    sharex=True,
    sharey="row",
)

for i, model in enumerate(MODELS):
    for j, band in enumerate(BANDS):
        ax = axes[i, j]
        sub = df_draw_stability[
            (df_draw_stability["model"] == model) & (df_draw_stability["band"] == band)
        ]
        counts = sub["n_draws"].value_counts().reindex([1, 2, 3], fill_value=0)
        total = counts.sum()
        fracs = counts / total if total > 0 else counts
        ax.bar(
            [1, 2, 3],
            fracs.values,
            color=BAND_COLORS.get(band, "#999999"),
            alpha=0.8,
            edgecolor="white",
        )
        ax.set_xticks([1, 2, 3])
        if i == len(MODELS) - 1:
            ax.set_xlabel("# Draws")
        if j == 0:
            ax.set_ylabel(f"{model}\nFraction")
        if i == 0:
            ax.set_title(BAND_NAMES.get(band, band))
        # Annotate counts
        for k_idx in range(3):
            ax.text(
                k_idx + 1,
                fracs.values[k_idx] + 0.02,
                f"{int(counts.iloc[k_idx])}",
                ha="center",
                fontsize=7,
            )

fig.suptitle("Per-Edge Draw Stability by Model and Band", fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, "deep_18_draw_stability_histograms.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_18_draw_stability_histograms.png


### 7b. Cross-Tabulation: Draw Stability x Band Sharing Level

Key question: are band-specific edges (sharing_level=1) also draw-unstable (n_draws < 3)?
If so, "band-specific" edges may be noise rather than genuine mechanisms.

In [38]:
# Cross-tabulate draw stability x band sharing level
all_cross_dfs = []
for model in MODELS:
    df_cross = compute_draw_stability_vs_sharing(
        circuits, model, reference_draw="draw_1"
    )
    if not df_cross.empty:
        all_cross_dfs.append(df_cross)

df_stab_vs_sharing = pd.concat(all_cross_dfs, ignore_index=True)
df_stab_vs_sharing.to_csv(ANALYSIS_DIR / "deep_stability_vs_sharing.csv", index=False)
print(f"Saved: {ANALYSIS_DIR / 'deep_stability_vs_sharing.csv'}")

# Print cross-tabulation per model
for model in MODELS:
    sub = df_stab_vs_sharing[
        (df_stab_vs_sharing["model"] == model)
        & df_stab_vs_sharing["sharing_level"].notna()
    ]
    ct = pd.crosstab(sub["n_draws"], sub["sharing_level"], margins=True)
    print(f"\n{model}: draw_stability (rows) x sharing_level (cols)")
    print(ct)

Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/analysis/deep_stability_vs_sharing.csv

pythia-70m: draw_stability (rows) x sharing_level (cols)
sharing_level  1.0  2.0  3.0  4.0   5.0   All
n_draws                                      
1              117   86   74   34     2   313
2               39   63   69  100    33   304
3               11   25   72  150  1450  1708
All            167  174  215  284  1485  2325

pythia-160m: draw_stability (rows) x sharing_level (cols)
sharing_level   1.0   2.0   3.0   4.0   5.0   All
n_draws                                          
1               924   751   439   238    34  2386
2               349   492   480   532   240  2093
3                64   184   322   716  3196  4482
All            1337  1427  1241  1486  3470  8961

pythia-410m: draw_stability (rows) x sharing_level (cols)
sharing_level   1.0   2.0   3.0   4.0   5.0    All
n_draws                                           
1              4411  2916  1606   628   114   9675


In [39]:
# Visualization: Cross-tabulation heatmaps (draw stability x sharing level)
fig, axes = plt.subplots(1, len(MODELS), figsize=(5.5 * len(MODELS), 4.5))

for idx, model in enumerate(MODELS):
    ax = axes[idx]
    sub = df_stab_vs_sharing[
        (df_stab_vs_sharing["model"] == model)
        & df_stab_vs_sharing["sharing_level"].notna()
    ]
    ct = pd.crosstab(sub["n_draws"], sub["sharing_level"])
    # Ensure all rows (1,2,3) and cols (1-5) present
    ct = ct.reindex(index=[1, 2, 3], columns=[1, 2, 3, 4, 5], fill_value=0)

    sns.heatmap(
        ct,
        ax=ax,
        cmap="YlOrRd",
        annot=True,
        fmt="d",
        square=True,
        linewidths=0,
        linecolor="none",
        cbar_kws={"label": "Edge Count"},
    )
    ax.set_xlabel("Sharing Level (# bands)")
    ax.set_ylabel("Draw Stability (# draws)")
    ax.set_title(model)

fig.suptitle("Draw Stability vs Band Sharing Level", fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, "deep_19_stability_vs_sharing_heatmap.png")

# Supplementary: bar chart of draw-stable fraction by sharing level
fig, axes = plt.subplots(1, len(MODELS), figsize=(4.5 * len(MODELS), 4), sharey=True)

for idx, model in enumerate(MODELS):
    ax = axes[idx]
    sub = df_stab_vs_sharing[
        (df_stab_vs_sharing["model"] == model)
        & df_stab_vs_sharing["sharing_level"].notna()
    ]
    # Fraction with n_draws == 3 per sharing level
    stable_frac = sub.groupby("sharing_level").apply(
        lambda g: (g["n_draws"] == 3).mean()
    )
    ax.bar(
        stable_frac.index,
        stable_frac.values,
        color="steelblue",
        alpha=0.8,
        edgecolor="white",
    )
    ax.set_xlabel("Sharing Level (# bands)")
    if idx == 0:
        ax.set_ylabel("Fraction Draw-Stable\n(all 3 draws)")
    ax.set_xticks([1, 2, 3, 4, 5])
    ax.set_title(model)
    ax.grid(True, alpha=0.3, axis="y")

fig.suptitle("Draw-Stable Fraction by Sharing Level", fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, "deep_20_stable_fraction_by_sharing.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_19_stability_vs_sharing_heatmap.png


  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_20_stable_fraction_by_sharing.png


### 7c. Reliable Band-Specific Edges

An edge is "reliably band-specific" if it is unique to exactly 1 band (sharing_level=1) AND present in all 3 draws (n_draws=3). These are the strongest candidates for genuine frequency-selective mechanisms.

In [40]:
# Identify reliable band-specific edges
reliable_bs = df_stab_vs_sharing[
    (df_stab_vs_sharing["sharing_level"] == 1) & (df_stab_vs_sharing["n_draws"] == 3)
].copy()

reliable_bs.to_csv(ANALYSIS_DIR / "deep_reliable_band_specific.csv", index=False)
print(f"Saved: {ANALYSIS_DIR / 'deep_reliable_band_specific.csv'}")
print(f"\nReliable band-specific edges per model x band:")
print(reliable_bs.groupby(["model", "band"]).size().unstack(fill_value=0))

# Compare to total band-specific edges
total_bs = df_stab_vs_sharing[df_stab_vs_sharing["sharing_level"] == 1]
for model in MODELS:
    total = len(total_bs[total_bs["model"] == model])
    reliable = len(reliable_bs[reliable_bs["model"] == model])
    pct = 100 * reliable / total if total > 0 else 0
    print(
        f"  {model}: {reliable}/{total} band-specific edges are draw-stable ({pct:.1f}%)"
    )

Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/analysis/deep_reliable_band_specific.csv

Reliable band-specific edges per model x band:
band         control  high  low  medium  very_high
model                                             
pythia-1.4b       15     9   59      27          6
pythia-160m       10     8   13      19         14
pythia-1b          3     0    9       5          7
pythia-410m       24    30  113      33         23
pythia-70m         3     2    2       1          3
  pythia-70m: 11/167 band-specific edges are draw-stable (6.6%)
  pythia-160m: 64/1337 band-specific edges are draw-stable (4.8%)
  pythia-410m: 223/6028 band-specific edges are draw-stable (3.7%)
  pythia-1b: 24/1251 band-specific edges are draw-stable (1.9%)
  pythia-1.4b: 116/4514 band-specific edges are draw-stable (2.6%)


### 7d. Universal Core Connectivity (S-G4)

Quick check: does a path exist from `embed` to the final residual using ONLY universal edges? Is the universal core one connected component or fragmented?

In [41]:
# Compute universal core connectivity for each model
core_rows = []
for model in MODELS:
    result = compute_universal_core_connectivity(circuits, model)
    core_rows.append(result)
    path_str = "YES" if result["path_exists"] else "NO"
    print(f"\n{model}:")
    print(
        f"  embed -> R{MODEL_INFO[model]['n_layers'] - 1} path exists (all draws): {path_str}"
    )
    print(f"  Path per draw: {result['path_per_draw']}")
    print(f"  Weakly connected components (mean): {result['n_components_mean']:.1f}")
    print(f"  Components per draw: {result['n_components_per_draw']}")
    print(
        f"  Node coverage (core/full): {result['n_nodes_core']}/{result['n_nodes_full']} "
        f"= {result['node_coverage']:.3f}"
    )
    print(
        f"  Edge coverage (core/full avg): {result['n_edges_core_mean']:.0f}/"
        f"{result['n_edges_full_mean']:.0f}"
    )

# Save summary table
df_core = pd.DataFrame(
    [
        {
            "model": r["model"],
            "path_exists_all_draws": r["path_exists"],
            "path_exists_any_draw": r["path_exists_any_draw"],
            "n_components_mean": r["n_components_mean"],
            "n_nodes_core": r["n_nodes_core"],
            "n_nodes_full": r["n_nodes_full"],
            "node_coverage": r["node_coverage"],
            "n_edges_core_mean": r["n_edges_core_mean"],
            "n_edges_full_mean": r["n_edges_full_mean"],
        }
        for r in core_rows
    ]
)
df_core.to_csv(ANALYSIS_DIR / "deep_universal_core_connectivity.csv", index=False)
print(f"\nSaved: {ANALYSIS_DIR / 'deep_universal_core_connectivity.csv'}")


pythia-70m:
  embed -> R5 path exists (all draws): YES
  Path per draw: {'draw_1': True, 'draw_2': True, 'draw_3': True}
  Weakly connected components (mean): 1.0
  Components per draw: {'draw_1': 1, 'draw_2': 1, 'draw_3': 1}
  Node coverage (core/full): 51/56 = 0.911
  Edge coverage (core/full avg): 302/533

pythia-160m:
  embed -> R11 path exists (all draws): YES
  Path per draw: {'draw_1': True, 'draw_2': True, 'draw_3': True}
  Weakly connected components (mean): 1.0
  Components per draw: {'draw_1': 1, 'draw_2': 1, 'draw_3': 1}
  Node coverage (core/full): 123/148 = 0.831
  Edge coverage (core/full avg): 713/2331



pythia-410m:
  embed -> R23 path exists (all draws): YES
  Path per draw: {'draw_1': True, 'draw_2': True, 'draw_3': True}
  Weakly connected components (mean): 1.0
  Components per draw: {'draw_1': 1, 'draw_2': 1, 'draw_3': 1}
  Node coverage (core/full): 249/366 = 0.680
  Edge coverage (core/full avg): 1303/7204



pythia-1b:
  embed -> R15 path exists (all draws): YES
  Path per draw: {'draw_1': True, 'draw_2': True, 'draw_3': True}
  Weakly connected components (mean): 1.0
  Components per draw: {'draw_1': 1, 'draw_2': 1, 'draw_3': 1}
  Node coverage (core/full): 92/143 = 0.643
  Edge coverage (core/full avg): 356/1730

pythia-1.4b:
  embed -> R23 path exists (all draws): YES
  Path per draw: {'draw_1': True, 'draw_2': True, 'draw_3': True}
  Weakly connected components (mean): 1.0
  Components per draw: {'draw_1': 1, 'draw_2': 1, 'draw_3': 1}
  Node coverage (core/full): 166/344 = 0.483
  Edge coverage (core/full avg): 620/5114

Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/analysis/deep_universal_core_connectivity.csv


### 7e. Skip Distance by Sharing Level (S-G5)

The sharing profiles in section 2a already computed `mean_layer_distance` per sharing level. Here we save the raw per-edge sharing data for the inferential test in NB04 and add a full distribution visualization.

In [42]:
# df_all_sharing is already in memory from cell 9 (compute_edge_sharing_with_properties)
# Save it so NB04 can load per-edge data for statistical tests
df_all_sharing.to_csv(ANALYSIS_DIR / "deep_edge_sharing_raw.csv", index=False)
print(f"Saved: {ANALYSIS_DIR / 'deep_edge_sharing_raw.csv'}")
print(f"  Rows: {len(df_all_sharing)}, Columns: {list(df_all_sharing.columns)}")

# Visualization: Box plot of layer_distance by sharing_level per model
fig, axes = plt.subplots(1, len(MODELS), figsize=(5 * len(MODELS), 4.5), sharey=True)

for idx, model in enumerate(MODELS):
    ax = axes[idx]
    sub = df_all_sharing[
        (df_all_sharing["model"] == model) & (df_all_sharing["draw"] == "draw_1")
    ]

    sharing_levels = sorted(sub["sharing_level"].unique())
    data_by_level = [
        sub[sub["sharing_level"] == sl]["layer_distance"].values
        for sl in sharing_levels
    ]
    bp = ax.boxplot(
        data_by_level,
        positions=sharing_levels,
        widths=0.6,
        patch_artist=True,
        showfliers=False,
    )
    for patch in bp["boxes"]:
        patch.set_facecolor("steelblue")
        patch.set_alpha(0.7)

    # Overlay medians
    medians = [np.median(d) if len(d) > 0 else 0 for d in data_by_level]
    ax.plot(sharing_levels, medians, "ro-", markersize=6, label="Median")

    ax.set_xlabel("Sharing Level (# bands)")
    if idx == 0:
        ax.set_ylabel("Layer Distance")
    ax.set_title(model)
    ax.set_xticks(sharing_levels)
    ax.grid(True, alpha=0.3, axis="y")

fig.suptitle("Edge Layer Distance by Sharing Level", fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, "deep_21_skip_distance_by_sharing.png")

Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/analysis/deep_edge_sharing_raw.csv
  Rows: 50735, Columns: ['raw', 'sharing_level', 'dst_type', 'dst_layer', 'src_type', 'src_layer', 'edge_category', 'layer_distance', 'is_skip', 'is_input', 'is_output', 'bands_present', 'model', 'draw', 'component']


  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/deep_21_skip_distance_by_sharing.png


---
## 8. Summary & Export

In [43]:
# Build master summary combining key deep metrics per circuit
master_rows = []

for _, row in df.iterrows():
    model, band, draw = row["model"], row["band"], row["draw"]
    cid = row["circuit_id"]

    master = {"circuit_id": cid, "model": model, "band": band, "draw": draw}

    # Graph metrics
    g_row = df_graph[
        (df_graph["model"] == model)
        & (df_graph["band"] == band)
        & (df_graph["draw"] == draw)
    ]
    if not g_row.empty:
        g = g_row.iloc[0]
        master["diameter"] = g["diameter"]
        master["clustering_coefficient"] = g["clustering_coefficient"]
        master["density"] = g["density"]
        master["avg_path_length"] = g["avg_path_length"]
        master["n_weakly_connected"] = g["n_weakly_connected"]

    master_rows.append(master)

df_master = pd.DataFrame(master_rows)
df_master.to_csv(ANALYSIS_DIR / "deep_master_summary.csv", index=False)
print(f"Saved: {ANALYSIS_DIR / 'deep_master_summary.csv'}")

# List all output files
print("\n=== All Deep Analysis Outputs ===")
print("\nCSV files:")
for f in sorted(ANALYSIS_DIR.glob("deep_*.csv")):
    print(f"  {f.name}")
print("\nVisualization files:")
for f in sorted(VIZ_DIR.glob("deep_*.png")):
    print(f"  {f.name}")

Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/analysis/deep_master_summary.csv

=== All Deep Analysis Outputs ===

CSV files:
  deep_all_hypothesis_tests.csv
  deep_band_affinity.csv
  deep_band_signatures.csv
  deep_component_jaccard.csv
  deep_component_wiring.csv
  deep_degree_stats.csv
  deep_directed_containment.csv
  deep_draw_stability.csv
  deep_edge_sharing_raw.csv
  deep_graph_metrics.csv
  deep_head_clusters.csv
  deep_head_entropy.csv
  deep_head_universality.csv
  deep_hub_nodes.csv
  deep_input_edges.csv
  deep_layer_sensitivity.csv
  deep_layer_universal_fraction.csv
  deep_master_summary.csv
  deep_output_edges.csv
  deep_reliable_band_specific.csv
  deep_sharing_by_component.csv
  deep_sharing_profiles.csv
  deep_stability_vs_sharing.csv
  deep_universal_core_connectivity.csv

Visualization files:
  deep_01_component_jaccard.png
  deep_02_sharing_by_component.png
  deep_03_component_wiring.png
  deep_04_sharing_profiles.png
  deep_05_band_affinity.png
  deep_

In [44]:
print("\n=== Key Findings Summary ===")
print("\n1. COMPONENT-LEVEL JACCARD:")
for model in MODELS:
    sub = df_comp_jaccard[df_comp_jaccard["model"] == model]
    max_gap_comp = sub.loc[sub["gap"].idxmax(), "component"]
    max_gap = sub["gap"].max()
    print(f"  {model}: largest within-between gap in {max_gap_comp} ({max_gap:+.4f})")

print("\n2. LAYER SENSITIVITY:")
for model in MODELS:
    sub = df_layer_sensitivity[df_layer_sensitivity["model"] == model]
    min_layer = sub.loc[sub["mean_jaccard"].idxmin()]
    max_layer = sub.loc[sub["mean_jaccard"].idxmax()]
    print(
        f"  {model}: most sensitive L{int(min_layer['layer'])} (J={min_layer['mean_jaccard']:.3f}), "
        f"least sensitive L{int(max_layer['layer'])} (J={max_layer['mean_jaccard']:.3f})"
    )

print("\n3. GRAPH METRICS:")
for model in MODELS:
    sub = df_graph[df_graph["model"] == model]
    print(
        f"  {model}: diameter={sub['diameter'].mean():.1f}, "
        f"clustering={sub['clustering_coefficient'].mean():.3f}, "
        f"WCCs={sub['n_weakly_connected'].mean():.1f}"
    )

print("\n4. HUB UNIVERSALITY:")
for model in MODELS:
    sub = df_all_hubs[
        (df_all_hubs["model"] == model)
        & (df_all_hubs["centrality_type"] == "betweenness")
    ]
    if not sub.empty:
        n_univ = sub["is_universal_hub"].sum()
        print(f"  {model}: {n_univ}/{len(sub)} betweenness hubs are universal")


=== Key Findings Summary ===

1. COMPONENT-LEVEL JACCARD:
  pythia-70m: largest within-between gap in attn (+0.0423)
  pythia-160m: largest within-between gap in attn (+0.0378)
  pythia-410m: largest within-between gap in resid (+0.0249)
  pythia-1b: largest within-between gap in attn (+0.0140)
  pythia-1.4b: largest within-between gap in resid (+0.0293)

2. LAYER SENSITIVITY:
  pythia-70m: most sensitive L2 (J=0.657), least sensitive L0 (J=1.000)
  pythia-160m: most sensitive L3 (J=0.423), least sensitive L11 (J=0.818)
  pythia-410m: most sensitive L3 (J=0.293), least sensitive L23 (J=0.771)
  pythia-1b: most sensitive L12 (J=0.322), least sensitive L0 (J=0.674)
  pythia-1.4b: most sensitive L20 (J=0.165), least sensitive L23 (J=0.640)

3. GRAPH METRICS:
  pythia-70m: diameter=2.5, clustering=0.713, WCCs=1.0
  pythia-160m: diameter=3.3, clustering=0.703, WCCs=1.0
  pythia-410m: diameter=3.9, clustering=0.687, WCCs=1.0
  pythia-1b: diameter=3.9, clustering=0.562, WCCs=1.0
  pythia-1.4